# 04. PSD and P80 Estimation from Point Cloud

Estimate fragment volumes from DBSCAN clusters, compute estimated PSD, and compare against synthetic ground truth.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from fragmentation.volume_estimation import estimate_cluster_volumes_convex_hull
from fragmentation.psd import equivalent_spherical_diameter, cumulative_psd, percentile_size
from fragmentation.p80 import p80_error_mm, p80_relative_error_pct
from visualisation.plot_psd import plot_psd

LABEL_DIR = ROOT / 'data' / 'labels'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'
for path in [FIGURE_DIR, TABLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

seg = np.load(LABEL_DIR / 'synthetic_rockpile_exterior_dbscan_segmentation.npz')
points = seg['points_xyz']
predicted_labels = seg['predicted_cluster_ids']
gt_fragments = pd.read_csv(LABEL_DIR / 'reference_fragments.csv')
gt_psd = pd.read_csv(TABLE_DIR / 'ground_truth_psd.csv')
gt_summary = pd.read_csv(TABLE_DIR / 'ground_truth_summary.csv').iloc[0]
if 'diameter_mm' not in gt_psd.columns and 'diameter_from_volume_mm' in gt_psd.columns:
    gt_psd = gt_psd.rename(columns={'diameter_from_volume_mm': 'diameter_mm'})
print('Predicted labels:', len(np.unique(predicted_labels[predicted_labels >= 0])))

In [ ]:
cluster_volumes = estimate_cluster_volumes_convex_hull(points, predicted_labels)
cluster_volumes = cluster_volumes[(cluster_volumes['n_points'] >= 25) & (cluster_volumes['estimated_volume_m3'] > 1e-7)].copy()
cluster_volumes['estimated_diameter_m'] = equivalent_spherical_diameter(cluster_volumes['estimated_volume_m3'].to_numpy())
estimated_psd = cumulative_psd(cluster_volumes['estimated_diameter_m'].to_numpy(), cluster_volumes['estimated_volume_m3'].to_numpy())

estimated_p10 = percentile_size(estimated_psd, 10.0) if len(estimated_psd) else np.nan
estimated_p50 = percentile_size(estimated_psd, 50.0) if len(estimated_psd) else np.nan
estimated_p80 = percentile_size(estimated_psd, 80.0) if len(estimated_psd) else np.nan
gt_p80 = float(gt_summary['P80_mm'])
comparison = pd.DataFrame([{
    'ground_truth_P10_mm': float(gt_summary['P10_mm']),
    'ground_truth_P50_mm': float(gt_summary['P50_mm']),
    'ground_truth_P80_mm': gt_p80,
    'estimated_P10_mm': estimated_p10,
    'estimated_P50_mm': estimated_p50,
    'estimated_P80_mm': estimated_p80,
    'P80_error_mm': p80_error_mm(estimated_p80, gt_p80) if np.isfinite(estimated_p80) else np.nan,
    'P80_relative_error_pct': p80_relative_error_pct(estimated_p80, gt_p80) if np.isfinite(estimated_p80) else np.nan,
    'n_estimated_fragments': int(len(cluster_volumes)),
    'n_ground_truth_fragments': int(len(gt_fragments)),
}])

cluster_path = TABLE_DIR / 'estimated_cluster_volumes.csv'
estimated_psd_path = TABLE_DIR / 'estimated_psd_dbscan.csv'
comparison_path = TABLE_DIR / 'p80_comparison.csv'
cluster_volumes.to_csv(cluster_path, index=False)
estimated_psd.to_csv(estimated_psd_path, index=False)
comparison.to_csv(comparison_path, index=False)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
plot_psd(gt_psd, ax=ax, label='Ground truth', marker='o')
if len(estimated_psd):
    plot_psd(estimated_psd, ax=ax, label='Estimated from DBSCAN clusters', marker='s')
ax.set_title('Estimated vs ground-truth PSD')
figure_path = FIGURE_DIR / 'synthetic_rockpile_estimated_vs_ground_truth_psd.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print(figure_path)